# ICICV-2027 — Flight-disjoint, persistence-referenced evaluation of camera-aided UAV mmWave beam prediction
**Data:** DeepSense 6G Scenario 23 (real 60 GHz drone measurements). **Accelerator:** GPU T4 (Settings → Accelerator). Internet ON is only needed if torchvision weights are not cached.

What this notebook does:
1. Loads Scenario 23 (images, 64-beam power → 32 beams, drone GPS/height/distance/speed/attitude, flight index).
2. Image features: (a) **drone localisation by background subtraction** (centroid, size: lightweight geometric cue), (b) frozen ImageNet ResNet-18 embedding (PCA-32).
3. Models for horizons h = 0, 1, 3, 5 (past beams known up to t-1): persistence, GPS, GPS+history, vision (embedding), fusion, end-to-end fine-tuned ResNet-18, and the proposed **persistence-anchored residual (PAR)** predictor (predicts the beam change Δb from image motion cues + GPS + beam history).
4. Two protocols: **random 5-fold over windows** (as in prior work) vs **flight-disjoint GroupKFold (5 folds)**.
5. Cost (parameters, latency) and robustness to natural image degradation (low light, blur, occlusion, dropped frames).

Outputs in `/kaggle/working/out/`: `results_main.csv`, `per_flight.csv`, `robustness.csv`, `cost.csv`, `summary.json`, `report.txt`, figures. **Send back `report.txt`, `summary.json` and the figures.**

In [ ]:
import os, glob, json, time, math, warnings, random
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.stats import wilcoxon
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold, GroupKFold
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

FAST = False            # True = quick smoke test (fewer folds/iterations)
Q = 32                  # 64 -> 32 beams (as in prior Scenario 23 work)
HORIZONS = [0, 1, 3, 5] # predict b[t+h]; beams known up to t-1 (one-slot feedback delay)
W = 8                   # history window (frames)
NFOLD = 3 if FAST else 5
D = 8                   # PAR: max |beam change| modelled explicitly (larger changes clipped; share reported)
GBT_ITERS = 40 if FAST else 150
CNN_EPOCHS = 2 if FAST else 6
CNN_HORIZONS = [0, 3]   # fine-tuned CNN only for these horizons (cost)
IMG_RES = 160
SEED = 0
OUT = "/kaggle/working/out" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
REPORT = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); REPORT.append(s)
def save_report():
    open(f"{OUT}/report.txt", "w").write("\n".join(REPORT))
from concurrent.futures import ThreadPoolExecutor
def pmap(f, xs, workers=32):
    with ThreadPoolExecutor(workers) as ex: return list(ex.map(f, xs))

# ImageNet weights: use an uploaded resnet18-*.pth from /kaggle/input if present, otherwise download (Internet ON)
import torch, torchvision, shutil
_ckdir = os.path.join(torch.hub.get_dir(), "checkpoints"); os.makedirs(_ckdir, exist_ok=True)
_w = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True) if "resnet18" in os.path.basename(p).lower()]
if _w:
    shutil.copy(_w[0], os.path.join(_ckdir, "resnet18-f37072fd.pth")); log("ResNet-18 weights copied from", _w[0])
try:
    torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT); log("ImageNet weights OK")
except Exception as e:
    raise RuntimeError("ResNet-18 weights not found: upload resnet18-f37072fd.pth as a Kaggle dataset and Add Input, or turn Internet ON") from e

## 1. Load Scenario 23

In [ ]:
cands = sorted(set(glob.glob("/kaggle/input/**/*.csv", recursive=True) + glob.glob("./**/scenario23*.csv", recursive=True)),
               key=lambda p: (("scenario23" in os.path.basename(p).lower()) * 10) - len(p) / 1000, reverse=True)
CSV = None
for p in cands:
    try:
        cols = pd.read_csv(p, nrows=2).columns.str.lower()
        if any("pwr" in c or "power" in c for c in cols): CSV = p; break
    except Exception: pass
assert CSV, "Scenario 23 CSV not found under /kaggle/input"
BASE = os.path.dirname(CSV)
df = pd.read_csv(CSV)
log("CSV:", CSV, df.shape); log("columns:", list(df.columns))

_cache = {}
def resolve(rel):
    rel = str(rel).strip()
    for c in [os.path.join(BASE, rel), os.path.join(BASE, rel.lstrip("./")), rel]:
        if os.path.exists(c): return c
    if not _cache:
        for f in glob.glob(os.path.join(BASE, "**", "*"), recursive=True): _cache.setdefault(os.path.basename(f), f)
    return _cache.get(os.path.basename(rel))
def load_val(v):
    if isinstance(v, str) and (v.endswith(".txt") or "/" in v):
        p = resolve(v); return None if p is None else np.loadtxt(p).astype(float).ravel()
    try: return np.array([float(v)])
    except Exception: return None
def find_col(*keys, exclude=()):
    for c in df.columns:
        lc = c.lower()
        if all(k in lc for k in keys) and not any(e in lc for e in exclude): return c
    return None

c_pwr = find_col("pwr") or find_col("power"); c_rgb = find_col("rgb") or find_col("image")
c_loc = find_col("unit2", "loc") or find_col("loc", exclude=("unit1",)); c_seq = find_col("seq")
c_bs = find_col("unit1", "loc"); c_idx = find_col("index") or find_col("time")
extra = {k: find_col("unit2", k) or find_col(k, exclude=("unit1",)) for k in ["height", "distance", "speed", "pitch", "roll"]}
log("power:", c_pwr, "rgb:", c_rgb, "loc:", c_loc, "seq:", c_seq, "extra:", extra)

# order frames in time inside each flight
if c_seq is not None and c_idx is not None:
    df = df.sort_values([c_seq, c_idx], kind="stable").reset_index(drop=True)

t0 = time.time(); P64 = np.stack(pmap(load_val, df[c_pwr].tolist()))
P = P64[:, ::2] if (P64.shape[1] == 64 and Q == 32) else P64
best = P.argmax(1)
loc = np.stack([x[:2] for x in pmap(load_val, df[c_loc].tolist())])
lat0, lon0 = loc[:, 0].mean(), loc[:, 1].mean()
if c_bs is not None:
    b0 = load_val(df[c_bs].iloc[0]); lat0, lon0 = b0[0], b0[1]
R_E = 6371000.0
Fd = {"east": np.deg2rad(loc[:, 1] - lon0) * R_E * np.cos(np.deg2rad(lat0)), "north": np.deg2rad(loc[:, 0] - lat0) * R_E}
for k, c in extra.items():
    if c is not None:
        Fd[k] = np.array([x[0] if x is not None else np.nan for x in pmap(load_val, df[c].tolist())])
F = pd.DataFrame(Fd)
F["az"] = np.degrees(np.arctan2(F["east"], F["north"]))
if "height" in F and "distance" in F: F["elev"] = np.degrees(np.arctan2(F["height"], F["distance"].clip(lower=1)))
TEL_COLS = list(F.columns)
seq = df[c_seq].values if c_seq else np.cumsum(np.r_[0, np.diff(np.arange(len(df))) != 1])
_, seq = np.unique(seq, return_inverse=True)
img_paths = pmap(resolve, df[c_rgb].tolist())
log(f"txt files parsed in {time.time()-t0:.0f} s")
N, NS = len(df), seq.max() + 1
lens = np.bincount(seq)
log(f"N={N} frames, flights={NS}, flight length min/median/max {lens.min()}/{int(np.median(lens))}/{lens.max()}, images found {sum(p is not None for p in img_paths)}")
ok = seq[1:] == seq[:-1]
log(f"P(beam changes between consecutive frames) = {(best[1:] != best[:-1])[ok].mean():.3f}")

## 2. Image features: drone localisation (background subtraction) and frozen ResNet-18 embedding

In [ ]:
import cv2
def read_small(p):
    im = cv2.imread(p, cv2.IMREAD_REDUCED_COLOR_2) if p else None     # 480x270 for 960x540 input
    if im is None: return np.zeros((270, 480, 3), np.uint8)
    if im.shape[:2] != (270, 480): im = cv2.resize(im, (480, 270), interpolation=cv2.INTER_AREA)
    return im
t0 = time.time(); IMG480 = np.stack(pmap(read_small, img_paths, 16))
log(f"images read in {time.time()-t0:.0f} s, array {IMG480.shape}, {IMG480.nbytes/1e9:.1f} GB")

def localise(IMGC):
    # moving-window median background (+-150 frames) -> |frame - bg| -> small blobs -> blob with largest
    # contrast mass, penalised by distance to the previous detection (simple tracking). Output (cx, cy, area, valid)
    G = np.full((N, 4), np.nan, np.float32); G[:, 2:] = 0
    for s in range(NS):
        ids = np.where(seq == s)[0]
        gray = np.stack([cv2.cvtColor(IMGC[i], cv2.COLOR_BGR2GRAY) for i in ids]).astype(np.float32)
        L, H, Wd = gray.shape; prev = None
        for c0 in range(0, L, 25):
            lo, hi = max(0, c0 - 150), min(L, c0 + 175)
            bg = np.median(gray[lo:hi:5], 0)
            for j in range(c0, min(L, c0 + 25)):
                d = cv2.GaussianBlur(np.abs(gray[j] - bg), (5, 5), 0)
                lab, n = ndimage.label(d > max(d.mean() + 5 * d.std(), 8.0))
                if n == 0: continue
                idx = np.arange(1, n + 1)
                area = np.asarray(ndimage.sum(np.ones_like(d), lab, idx))
                mass = np.asarray(ndimage.sum(d, lab, idx))
                ok = np.where((area >= 2) & (area <= 0.003 * H * Wd))[0]
                if len(ok) == 0: continue
                com = np.array(ndimage.center_of_mass(d, lab, idx[ok]))      # (k, 2) rows: (cy, cx)
                cx, cy = com[:, 1] / Wd, com[:, 0] / H
                score = mass[ok] if prev is None else mass[ok] / (1 + 20 * np.hypot(cx - prev[0], cy - prev[1]))
                k = int(np.argmax(score))
                G[ids[j]] = [cx[k], cy[k], area[ok][k] / (H * Wd), 1.0]; prev = (cx[k], cy[k])
    for s in range(NS):                          # causal fill of missed detections
        last = None
        for i in np.where(seq == s)[0]:
            if G[i, 3] == 1: last = G[i, :3].copy()
            elif last is not None: G[i, :3] = last
    return np.nan_to_num(G, nan=0.5)

from scipy.stats import spearmanr
t0 = time.time(); G_clean = localise(IMG480)
v = G_clean[:, 3] == 1
r = spearmanr(G_clean[v, 0], best[v]).correlation
log(f"localisation done in {time.time()-t0:.0f} s, detection rate {v.mean():.3f}, Spearman(x-centroid, beam) = {r:.3f}, Spearman(y-centroid, beam) = {spearmanr(G_clean[v,1], best[v]).correlation:.3f}")

fig, ax = plt.subplots(1, 4, figsize=(16, 3))
for a, i in zip(ax, np.linspace(0, N - 1, 4).astype(int)):
    im = cv2.cvtColor(IMG480[i], cv2.COLOR_BGR2RGB)
    a.imshow(im); a.scatter([G_clean[i,0]*im.shape[1]], [G_clean[i,1]*im.shape[0]], s=120, facecolors="none", edgecolors="r", linewidths=2)
    a.set_title(f"frame {i}, beam {best[i]}"); a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_localisation_examples.png", dpi=150); plt.close()
fig, ax = plt.subplots(figsize=(5, 4)); ax.scatter(G_clean[v, 0], best[v], s=1, alpha=.3)
ax.set_xlabel("drone x-centroid in image (normalised)"); ax.set_ylabel("optimal beam index"); plt.tight_layout()
plt.savefig(f"{OUT}/fig_centroid_vs_beam.png", dpi=200); plt.close()

In [ ]:
DEV = "cuda" if torch.cuda.is_available() else "cpu"
log("device:", DEV)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def small_rgb(IMGC):
    return np.stack([cv2.cvtColor(cv2.resize(im, (IMG_RES, IMG_RES), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB) for im in IMGC])
def to_tensor(x): return ((torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255.) - MEAN) / STD
def resnet18(pretrained=True):
    return torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None)

@torch.no_grad()
def embed(IMG):
    net = resnet18(); net.fc = torch.nn.Identity(); net.eval().to(DEV)
    return np.concatenate([net(to_tensor(IMG[i:i+256]).to(DEV)).cpu().numpy() for i in range(0, len(IMG), 256)])

t0 = time.time(); IMG_clean = small_rgb(IMG480); E_raw = embed(IMG_clean)
log(f"small images + embedding in {time.time()-t0:.0f} s, emb {E_raw.shape}")

## 3. Samples, feature sets and models

In [ ]:
def build_index(h):
    # rows: (t, target frame t+h); needs W past frames for history
    T, TG = [], []
    for s in range(NS):
        ids = np.where(seq == s)[0]
        for j in range(W, len(ids) - h):
            T.append(ids[j]); TG.append(ids[j + h])
    return np.array(T), np.array(TG)

TEL = F[TEL_COLS].values.astype(float)
def feats(T, G, E):
    # returns dict of feature blocks (rows aligned with T)
    prev = T - 1                                       # last frame with a known beam
    hist_idx = np.stack([T - k for k in range(1, W + 1)], 1)   # t-1 ... t-W
    bh = best[hist_idx]                                # past optimal beams
    b_prev = best[prev]
    blk = {}
    blk["tel"] = TEL[T]
    blk["pos"] = TEL[T][:, POS_COLS]
    blk["tel_d"] = np.concatenate([TEL[T] - TEL[T - k] for k in (1, 3, W - 1)], 1)
    blk["beam_abs"] = bh.astype(float)
    blk["beam_rel"] = (bh - b_prev[:, None]).astype(float)       # history relative to last beam
    blk["b_prev"] = b_prev[:, None].astype(float)
    blk["img_geo"] = G[T]
    blk["img_geo_d"] = np.concatenate([G[T, :3] - G[T - k, :3] for k in (1, 3, W - 1)], 1)
    blk["emb"] = E[T]
    return blk, b_prev

FEATURE_SETS = {
    # name: (blocks, residual target?)
    "GPS-KNN":        (["pos"], False),
    "GPS":            (["tel"], False),
    "GPS+hist":       (["tel", "tel_d", "beam_abs"], False),
    "Vision(emb)":    (["emb"], False),
    "Fusion(emb+GPS)":(["emb", "tel"], False),
    "Fusion+hist":    (["emb", "tel", "tel_d", "beam_abs"], False),
    "PAR-noImg":      (["b_prev", "beam_rel", "tel", "tel_d"], True),
    "PAR (proposed)": (["b_prev", "beam_rel", "tel", "tel_d", "img_geo", "img_geo_d"], True),
}

try:
    import lightgbm as lgb; HAVE_LGB = True
except Exception:
    HAVE_LGB = False
log("tabular learner:", "LightGBM" if HAVE_LGB else "sklearn HGB (regularised)")
def gbt(seed):
    if HAVE_LGB:
        return lgb.LGBMClassifier(n_estimators=GBT_ITERS, learning_rate=0.1, num_leaves=31, min_child_samples=20,
                                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                  random_state=seed, verbose=-1, n_jobs=4)
    return HistGradientBoostingClassifier(max_iter=GBT_ITERS, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
                                          l2_regularization=1.0, early_stopping=False, random_state=seed)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
POS_COLS = [TEL_COLS.index(c) for c in ["east", "north", "height"] if c in TEL_COLS]
def knn(seed): return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, weights="distance"))

def scores_to_beams(proba, classes, b_prev, residual):
    S = np.zeros((len(proba), Q))
    if residual:
        for j, dlt in enumerate(classes):
            tgt = np.clip(b_prev + dlt, 0, Q - 1)
            np.add.at(S, (np.arange(len(proba)), tgt), proba[:, j])
    else:
        S[:, classes] = proba
    return S

def persistence_scores(b_prev):
    S = np.full((len(b_prev), Q), 1e-6)
    for d, w in [(0, 1.0), (1, .5), (-1, .45), (2, .2), (-2, .15)]:
        t = b_prev + d; m = (t >= 0) & (t < Q); S[np.where(m)[0], t[m]] = w
    return S

def metrics(S, y, tg):
    order = np.argsort(-S, 1)
    m = {f"top{k}": float(np.mean((order[:, :k] == y[:, None]).any(1))) for k in (1, 3, 5)}
    p_opt = P[tg, y]; p_hat = P[tg, order[:, 0]]
    pl = 10 * np.log10(np.maximum(p_opt, 1e-12) / np.maximum(p_hat, 1e-12))
    m["pl_db"] = float(pl.mean()); m["pl95_db"] = float(np.percentile(pl, 95))
    return m, (order[:, 0] == y)

## 4. Fine-tuned ResNet-18 baseline (end-to-end vision, as in prior work)

In [ ]:
def train_cnn(IMG, T_tr, y_tr, T_te, seed):
    torch.manual_seed(seed)
    net = resnet18(); net.fc = torch.nn.Linear(512, Q); net.to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(); bs = 64
    for ep in range(CNN_EPOCHS):
        net.train(); perm = np.random.default_rng(seed + ep).permutation(len(T_tr))
        for i in range(0, len(perm), bs):
            b = perm[i:i+bs]
            x = to_tensor(IMG[T_tr[b]]).to(DEV); yy = torch.from_numpy(y_tr[b]).long().to(DEV)
            opt.zero_grad(); lossf(net(x), yy).backward(); opt.step()
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(T_te), 256):
            out.append(torch.softmax(net(to_tensor(IMG[T_te[i:i+256]]).to(DEV)), 1).cpu().numpy())
    return np.concatenate(out), net

## 5. Main experiment: random vs flight-disjoint, all horizons

In [ ]:
rows, perflight = [], []
models_cache = {}
for h in HORIZONS:
    T, TG = build_index(h); y = best[TG]; groups = seq[T]
    dlt = y - best[T - 1]
    log(f"\n=== h={h}: {len(T)} samples; |Δb|>{D}: {(np.abs(dlt) > D).mean():.4f}; persistence correct {np.mean(dlt == 0):.3f}")
    for proto in ["random", "flight"]:
        splitter = KFold(NFOLD, shuffle=True, random_state=SEED).split(T) if proto == "random" else GroupKFold(NFOLD).split(T, groups=groups)
        for fold, (tr, te) in enumerate(splitter):
            # PCA of the embedding fitted on training frames only
            pca = PCA(32, random_state=SEED).fit(E_raw[np.unique(T[tr])])
            E = pca.transform(E_raw).astype(np.float32)
            blk, b_prev = feats(T, G_clean, E)
            res = {}
            S = persistence_scores(b_prev[te]); res["Persistence"] = metrics(S, y[te], TG[te])
            for name, (blocks, residual) in FEATURE_SETS.items():
                X = np.concatenate([blk[b] for b in blocks], 1)
                target = np.clip(dlt, -D, D) if residual else y
                clf = (knn if name == "GPS-KNN" else gbt)(SEED + fold).fit(X[tr], target[tr])
                S = scores_to_beams(clf.predict_proba(X[te]), clf.classes_, b_prev[te], residual)
                res[name] = metrics(S, y[te], TG[te])
                if fold == 0:   # sanity check: training accuracy (should be high, not ~0.2)
                    Str = scores_to_beams(clf.predict_proba(X[tr]), clf.classes_, b_prev[tr], residual)
                    log(f"   sanity h={h} {proto} {name}: train top1 {metrics(Str, y[tr], TG[tr])[0]['top1']:.3f}, test top1 {res[name][0]['top1']:.3f}")
                if proto == "flight" and fold == 0: models_cache[(h, name)] = (clf, X[te][:256])
            if h in CNN_HORIZONS:
                proba, net = train_cnn(IMG_clean, T[tr], y[tr], T[te], SEED + fold)
                res["ResNet-18 (fine-tuned)"] = metrics(proba, y[te], TG[te])
                if proto == "flight" and fold == 0: models_cache[(h, "ResNet-18 (fine-tuned)")] = net
            for name, (m, correct) in res.items():
                rows.append({"h": h, "protocol": proto, "fold": fold, "model": name, **m})
                if proto == "flight":
                    for fl in np.unique(groups[te]):
                        mk = groups[te] == fl
                        perflight.append({"h": h, "model": name, "flight": int(fl), "n": int(mk.sum()), "top1": float(correct[mk].mean())})
            log(f"h={h} {proto} fold {fold}: " + ", ".join(f"{k}={v[0]['top1']:.3f}" for k, v in res.items()))
            pd.DataFrame(rows).to_csv(f"{OUT}/results_main.csv", index=False); pd.DataFrame(perflight).to_csv(f"{OUT}/per_flight.csv", index=False); save_report()

R = pd.DataFrame(rows); PF = pd.DataFrame(perflight); PF.to_csv(f"{OUT}/per_flight.csv", index=False)

## 6. Summary tables, leakage gap and paired tests

In [ ]:
agg = R.groupby(["h", "protocol", "model"])[["top1", "top3", "top5", "pl_db"]].agg(["mean", "std"])
log("\n" + agg.round(3).to_string())
piv = R.groupby(["h", "model", "protocol"])["top1"].mean().unstack("protocol")
piv["leak_gap_pp"] = 100 * (piv["random"] - piv["flight"])
log("\nTop-1, random vs flight-disjoint (leakage gap in pp):\n" + piv.round(3).to_string())

# paired Wilcoxon over flights (flight-disjoint protocol): PAR vs each model
tests = []
for h in HORIZONS:
    a = PF[(PF.h == h) & (PF.model == "PAR (proposed)")].set_index("flight")["top1"]
    for m in PF[PF.h == h].model.unique():
        if m == "PAR (proposed)": continue
        b = PF[(PF.h == h) & (PF.model == m)].set_index("flight")["top1"]
        j = a.index.intersection(b.index); d = a[j] - b[j]
        p = wilcoxon(d).pvalue if (d != 0).sum() > 5 else np.nan
        tests.append({"h": h, "vs": m, "mean_diff_pp": 100 * d.mean(), "wins": int((d > 0).sum()), "losses": int((d < 0).sum()), "n_flights": len(j), "p": p})
TT = pd.DataFrame(tests); TT.to_csv(f"{OUT}/paired_tests.csv", index=False)
log("\nPAR vs others, per-flight top-1 (flight-disjoint):\n" + TT.round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, proto in zip(ax, ["random", "flight"]):
    sub = R[R.protocol == proto].groupby(["model", "h"])["top1"].mean().unstack("h")
    for m in sub.index: a.plot(sub.columns, sub.loc[m], marker="o", label=m)
    a.set_title(f"{proto} split"); a.set_xlabel("horizon h (frames)"); a.grid(alpha=.3)
ax[0].set_ylabel("top-1 accuracy"); ax[1].legend(fontsize=7, loc="lower left"); plt.tight_layout()
plt.savefig(f"{OUT}/fig_top1_vs_horizon.png", dpi=200); plt.close()
save_report()

## 7. Cost: parameters and single-sample latency

In [ ]:
cost = []
def gbt_size(clf):
    if HAVE_LGB and hasattr(clf, "booster_"):
        return int(sum(t["num_leaves"] * 2 - 1 for t in clf.booster_.dump_model()["tree_info"]))
    if hasattr(clf, "_predictors"):
        return int(sum(len(p.nodes) for it in clf._predictors for p in it))
    return int(clf[-1]._fit_X.size)   # KNN: stored training values
for (h, name), obj in models_cache.items():
    if h != 0: continue
    if name.startswith("ResNet"):
        net = obj; n_par = sum(p.numel() for p in net.parameters())
        x = to_tensor(IMG_clean[:1]).to(DEV)
        with torch.no_grad():
            for _ in range(10): net(x)
            if DEV == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(100): net(x)
            if DEV == "cuda": torch.cuda.synchronize()
        lat = (time.perf_counter() - t0) / 100 * 1e3
        cost.append({"model": name, "size": n_par, "unit": "parameters", "latency_ms": lat, "device": DEV})
    else:
        clf, Xs = obj
        t0 = time.perf_counter()
        for i in range(100): clf.predict_proba(Xs[i % len(Xs)][None])
        lat = (time.perf_counter() - t0) / 100 * 1e3
        cost.append({"model": name, "size": gbt_size(clf), "unit": "stored values" if name == "GPS-KNN" else "tree nodes", "latency_ms": lat, "device": "cpu"})
cost.append({"model": "ResNet-18 embedding (for Vision/Fusion GBT)", "size": 11176512, "unit": "parameters", "latency_ms": np.nan, "device": DEV})
C = pd.DataFrame(cost); C.to_csv(f"{OUT}/cost.csv", index=False); log("\nCost:\n" + C.to_string())
save_report()

## 8. Robustness to natural image degradation (flight-disjoint, h = 0 and 3)

In [ ]:
def low_light(im, rng):   # gamma darkening + sensor noise
    x = (im / 255.) ** 2.5 * 0.5; x = x + rng.normal(0, 0.03, x.shape); return (np.clip(x, 0, 1) * 255).astype(np.uint8)
def blur(im, rng): return cv2.GaussianBlur(im, (0, 0), 4)
def occlusion(im, rng):
    H, Wd = im.shape[:2]; h, w = H // 2, Wd // 2; y0, x0 = rng.integers(0, H - h), rng.integers(0, Wd - w)
    im = im.copy(); im[y0:y0+h, x0:x0+w] = 0; return im
DEGR = {"low-light": low_light, "blur": blur, "occlusion": occlusion}

rob = []
for dname, fn in list(DEGR.items()) + [("frame-drop 30%", None)]:
    if fn is not None:
        rng = np.random.default_rng(1); D480 = np.stack([fn(im, rng) for im in IMG480])
        G_d = localise(D480); IMG_d = small_rgb(D480); E_d_raw = embed(IMG_d); del D480
    else:   # dropped frames: image features frozen at the last delivered frame
        rng = np.random.default_rng(1); drop = rng.random(N) < 0.3
        keep = np.arange(N)
        for i in range(1, N):
            if drop[i] and seq[i] == seq[i - 1]: keep[i] = keep[i - 1]
        G_d = G_clean[keep]; E_d_raw = E_raw[keep]; IMG_d = IMG_clean[keep]
    for h in [0, 3]:
        T, TG = build_index(h); y = best[TG]; groups = seq[T]; dlt = y - best[T - 1]
        for fold, (tr, te) in enumerate(GroupKFold(NFOLD).split(T, groups=groups)):
            pca = PCA(32, random_state=SEED).fit(E_raw[np.unique(T[tr])])
            E_tr = pca.transform(E_raw).astype(np.float32); E_te = pca.transform(E_d_raw).astype(np.float32)
            blk_tr, b_prev = feats(T, G_clean, E_tr); blk_te, _ = feats(T, G_d, E_te)   # train clean, test degraded
            for name in ["Vision(emb)", "Fusion+hist", "PAR (proposed)"]:
                blocks, residual = FEATURE_SETS[name]
                Xtr = np.concatenate([blk_tr[b] for b in blocks], 1); Xte = np.concatenate([blk_te[b] for b in blocks], 1)
                target = np.clip(dlt, -D, D) if residual else y
                clf = gbt(SEED + fold).fit(Xtr[tr], target[tr])
                S = scores_to_beams(clf.predict_proba(Xte[te]), clf.classes_, b_prev[te], residual)
                rob.append({"degradation": dname, "h": h, "fold": fold, "model": name, **metrics(S, y[te], TG[te])[0]})
            if h in CNN_HORIZONS and fold < (2 if not FAST else 1):
                _, net = train_cnn(IMG_clean, T[tr], y[tr], T[te], SEED + fold)   # trained on clean images
                with torch.no_grad():
                    out = [torch.softmax(net(to_tensor(IMG_d[T[te][i:i+256]]).to(DEV)), 1).cpu().numpy() for i in range(0, len(te), 256)]
                rob.append({"degradation": dname, "h": h, "fold": fold, "model": "ResNet-18 (fine-tuned)", **metrics(np.concatenate(out), y[te], TG[te])[0]})
    log(f"robustness done: {dname}"); pd.DataFrame(rob).to_csv(f"{OUT}/robustness.csv", index=False)
RB = pd.DataFrame(rob)
clean = R[(R.protocol == "flight") & (R.h.isin([0, 3]))].groupby(["h", "model"])["top1"].mean().rename("clean")
rbs = RB.groupby(["h", "model", "degradation"])["top1"].mean().unstack("degradation").join(clean)
log("\nRobustness (top-1, flight-disjoint, trained on clean images):\n" + rbs.round(3).to_string())
save_report()

## 9. Save summary

In [ ]:
summary = {
    "csv": CSV, "N": int(N), "flights": int(NS), "Q": Q, "W": W, "folds": NFOLD, "horizons": HORIZONS,
    "detection_rate": float(G_clean[:, 3].mean()), "spearman_centroid_beam": float(r),
    "top1_mean": {f"h{h}|{p}|{m}": float(v) for (h, p, m), v in R.groupby(["h", "protocol", "model"])["top1"].mean().items()},
    "pl_db_mean": {f"h{h}|{p}|{m}": float(v) for (h, p, m), v in R.groupby(["h", "protocol", "model"])["pl_db"].mean().items()},
}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1); save_report()
print("DONE. Files in", OUT, os.listdir(OUT))